In [1]:
from pyiceberg.catalog import load_catalog
import duckdb

PROJECT = "/Users/Shared/academy/learner/projects/ethereum_iceberg"
catalog = load_catalog(
    "local",
    type="sql",
    uri=f"sqlite:///{PROJECT}/catalog/pyiceberg_catalog.db",
    warehouse=f"file://{PROJECT}/warehouse",
)
S3 = "s3://aws-public-blockchain/v1.0/eth"
con = duckdb.connect()
con.execute("set s3_region = 'us-east-2'")
catalog.list_namespaces()

[('raw',), ('ref',)]

In [2]:
catalog.list_tables("raw"), catalog.list_tables("ref")

([('raw', 'blocks')],
 [('ref', 'daily_block_stats'), ('ref', 'known_addresses')])

In [3]:
blocks = catalog.load_table("raw.blocks")
print(blocks.schema())
blocks.scan().to_arrow().num_rows

table {
  1: difficulty: optional double
  2: hash: optional string
  3: miner: optional string
  4: nonce: optional string
  5: number: optional long
  6: size: optional long
  7: timestamp: optional timestamp
  8: base_fee_per_gas: optional long
  9: gas_limit: optional long
  10: gas_used: optional long
  11: extra_data: optional string
  12: parent_hash: optional string
  13: state_root: optional string
  14: receipts_root: optional string
  15: transactions_root: optional string
  16: sha3_uncles: optional string
  17: transaction_count: optional long
  18: date: optional date
}


7177

In [4]:
def blocks_for(pattern, where="true"):
    return con.execute(f"""
        select * exclude (logs_bloom, last_modified)
        from read_parquet('{S3}/blocks/date={pattern}/*.parquet')
        where {where}
    """).to_arrow_table()

blocks.append(blocks_for("2026-08-*", "date > '2026-08-01'"))
blocks.append(blocks_for("2026-09-*"))
blocks.append(blocks_for("2026-10-*"))

In [5]:
blocks.scan().to_arrow().num_rows, len(blocks.snapshots())

(659896, 4)